# 06 Soil Profiles and Horizons

**Series:** Tribal Soils and Geology

## Soil Horizon Data From Surface to Bedrock

Soil profiles describe the vertical sequence of horizons from the surface
downward to bedrock or to the depth of investigation. Each horizon has
distinct physical and chemical properties that reflect how the soil
formed and how it behaves under use.

This notebook works at two levels:

**SSURGO horizon data (public):** The USDA NRCS chorizon table contains
horizon-level measurements for representative pedons in each map unit.
These data include texture, pH, organic matter, bulk density, available
water capacity, and more. However, SSURGO pedon sampling on reservation
lands is often sparse; one representative pedon may be used to
characterize a map unit covering thousands of hectares.

**Tribal-collected profiles (Tribal data):** The `soil_profile_template.xlsx`
provides a format for Tribal geoscientists and resource managers to collect
field profiles that supplement and refine the SSURGO data. This data
is gitignored and governed by OCAP®.

In [ ]:
# Imports
import sys
from pathlib import Path
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path: sys.path.insert(0, str(REPO_ROOT))
import warnings, numpy as np, pandas as pd
import geopandas as gpd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, contextily as ctx, yaml
from shapely.geometry import box
from src.constants import (
    CRS_GEOGRAPHIC, CRS_PROJECTED, CRS_WEB, REPO_ROOT as _REPO_ROOT,
    OUTPUTS_DIR, FIGURES_DIR, PINE_RIDGE_BBOX, ROSEBUD_BBOX,
    COMBINED_BBOX, STUDY_BBOX, WSD_3D_MODEL, WSD_STRATIGRAPHY, WSD_KEY_UNITS,
)
from src.loaders import load_tribal_boundaries
from src.sovereignty import print_data_acknowledgment, generate_citations, attach_provenance
warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline
with open(_REPO_ROOT/"config"/"config.yaml") as f: CONFIG = yaml.safe_load(f)
TEAL="#007A6E"; TEAL_LT="#E0F4F2"; GRAY="#566573"; TERRACOTTA="#C0392B"
def despine(ax):
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
primary = load_tribal_boundaries(["Pine Ridge","Rosebud"])
print(f"Ready. Nations: {len(primary)}")

In [ ]:
# Print data acknowledgement at the top of every notebook
print_data_acknowledgment(source_keys=["usda_ssurgo","tribal_soil_profiles"])

## SSURGO Horizon Data

In [ ]:
from src.loaders import load_ssurgo_horizons, load_tribal_soil_profiles

print("Loading SSURGO horizon (chorizon) data...")
horizons = load_ssurgo_horizons()

if horizons.empty:
    print("SSURGO not loaded. Run notebook 05 setup first.")
else:
    print(f"Horizon records: {len(horizons):,}")
    print(f"Columns: {horizons.columns.tolist()}")

    # Key physical properties
    NUMERIC_FIELDS = [
        "hzdept_r", "hzdepb_r", "sandtotal_r", "silttotal_r",
        "claytotal_r", "om_r", "ph1to1h2o_r", "cec7_r",
        "awc_r", "dbthirdbar_r", "ksat_r", "lep_r"
    ]
    available = [f for f in NUMERIC_FIELDS if f in horizons.columns]
    print(f"\nNumeric fields available: {available}")

    for field in available:
        horizons[field] = pd.to_numeric(horizons[field], errors="coerce")

    if available:
        print(f"\nHorizon depth statistics:")
        print(horizons[["hzdept_r","hzdepb_r"]].describe().round(1).to_string()
              if "hzdept_r" in horizons.columns else "depth columns not found")

In [ ]:
# Horizon texture

if not horizons.empty and "hzdept_r" in horizons.columns:
    # Texture triangle summary
    texture_fields = ["sandtotal_r","silttotal_r","claytotal_r"]
    if all(f in horizons.columns for f in texture_fields):
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # Left: texture by depth
        ax = axes[0]
        depth_bins = [0, 25, 50, 100, 150, 200]
        horizons["depth_class"] = pd.cut(horizons["hzdept_r"], bins=depth_bins,
                                          labels=["0-25","25-50","50-100","100-150","150-200"])
        depth_texture = horizons.groupby("depth_class")[texture_fields].mean()
        depth_texture.plot(kind="bar", ax=ax, color=["#F4D03F","#85C1E9","#7986CB"],
                           alpha=0.85, width=0.7)
        ax.set_xlabel("Depth class (cm)", fontsize=9)
        ax.set_ylabel("Mean percent (%)", fontsize=9)
        ax.set_title("Texture by Depth Class\n(SSURGO representative pedons)",
                     fontsize=10, fontweight="bold")
        ax.legend(["Sand","Silt","Clay"], fontsize=9)
        despine(ax)

        # Right: clay content distribution
        ax = axes[1]
        ax.hist(horizons["claytotal_r"].dropna(), bins=30,
                color="#7986CB", alpha=0.85, edgecolor="white")
        ax.axvline(horizons["claytotal_r"].median(), color="white",
                   linewidth=2, linestyle="--",
                   label=f"Median: {horizons['claytotal_r'].median():.1f}%")
        ax.set_xlabel("Clay content (%)", fontsize=9)
        ax.set_ylabel("Frequency", fontsize=9)
        ax.set_title("Clay Content Distribution\n(all SSURGO horizons in study area)",
                     fontsize=10, fontweight="bold")
        ax.legend(fontsize=9)
        despine(ax)

        plt.suptitle("SSURGO Horizon Texture for Pine Ridge and Rosebud Study Area",
                     fontsize=12, fontweight="bold")
        plt.tight_layout()
        fig.savefig(FIGURES_DIR/"06_horizon_texture.png", dpi=150, bbox_inches="tight")
        plt.show()

## Shrink-Swell Potential (LEP)

In [ ]:
# Linear extensibility percent (LEP) is the key indicator of expansive soil behavior
# LEP > 6% = moderate shrink-swell; LEP > 9% = high (NRCS criteria)
# Pierre Shale-derived soils routinely exceed 9%

if not horizons.empty and "lep_r" in horizons.columns:
    lep = pd.to_numeric(horizons["lep_r"], errors="coerce").dropna()
    lep_classes = pd.cut(lep,
                         bins=[0, 3, 6, 9, 100],
                         labels=["Low (<3%)", "Moderate (3-6%)",
                                 "High (6-9%)", "Very High (>9%)"])
    lep_dist = lep_classes.value_counts()

    print("SHRINK-SWELL POTENTIAL for Linear Extensibility Percent (LEP)")
    print("NRCS thresholds: Low <3% | Moderate 3-6% | High 6-9% | Very High >9%")
    print()
    total = lep_dist.sum()
    for cls, n in lep_dist.items():
        bar = chr(9608) * int(n/total*40)
        print(f"  {cls:<25}: {n:>5} horizons ({n/total*100:>4.1f}%) {bar}")
    print()
    high_lep = (lep >= 6).sum() / len(lep) * 100
    print(f"Horizons with LEP ≥ 6% (moderate or greater): {high_lep:.1f}%")
    print()
    print("High LEP horizons reflect Pierre Shale parent material.")
    print("Engineering structures placed on these soils without mitigation")
    print("experience progressive foundation movement and pavement failure.")
else:
    print("LEP data not available: load SSURGO horizon data first.")

## Tribal-Collected Soil Profiles

In [ ]:
print("Loading Tribal-collected soil profiles...")
tribal_profiles = load_tribal_soil_profiles()

if tribal_profiles.empty:
    print()
    print("No Tribal soil profile data found.")
    print()
    print("To add Tribal-collected profiles:")
    print("  1. Copy data/templates/soil_profile_template.xlsx")
    print("  2. Fill in field measurements")
    print("  3. Save as data/raw/soil_profiles.xlsx")
    print("  4. Re-run this cell")
    print()
    print("The template is designed to match the SSURGO schema,")
    print("so Tribal profiles can be directly compared to SSURGO data.")
else:
    print(f"Tribal profiles loaded: {len(tribal_profiles)} records")
    print(f"Profile IDs: {tribal_profiles['profile_id'].unique().tolist()}")
    print()
    # Compare Tribal profile pH to SSURGO
    if "pH" in tribal_profiles.columns and not horizons.empty and "ph1to1h2o_r" in horizons.columns:
        tribal_ph = pd.to_numeric(tribal_profiles["pH"], errors="coerce").dropna()
        ssurgo_ph = pd.to_numeric(horizons["ph1to1h2o_r"], errors="coerce").dropna()
        print("pH COMPARISON between Tribal-collected data and SSURGO")
        print(f"  Tribal profiles: mean={tribal_ph.mean():.2f}, n={len(tribal_ph)}")
        print(f"  SSURGO:          mean={ssurgo_ph.mean():.2f}, n={len(ssurgo_ph)}")

In [ ]:
# Print citations
print(generate_citations(["usda_ssurgo","tribal_soil_profiles"]))